In [ ]:
# Data Recovery Using Time Travel
# Demonstrates recovering accidentally deleted records using Snowflake Time Travel

In [ ]:
%%sql -r use_ctx
USE DATABASE SNOWFLAKE_ASSIGNMENT;
USE SCHEMA ASSIGNMENT_SCHEMA;

In [ ]:
%%sql -r create_table
-- Create a table with sample product inventory data
CREATE OR REPLACE TABLE PRODUCT_INVENTORY (
    PRODUCT_ID INT,
    PRODUCT_NAME VARCHAR(100),
    CATEGORY VARCHAR(50),
    STOCK_QTY INT,
    PRICE DECIMAL(10,2)
) DATA_RETENTION_TIME_IN_DAYS = 1;

In [ ]:
%%sql -r insert_data
-- Insert sample records
INSERT INTO PRODUCT_INVENTORY VALUES
(1, 'Wireless Mouse', 'Electronics', 150, 29.99),
(2, 'Mechanical Keyboard', 'Electronics', 80, 89.99),
(3, 'USB-C Hub', 'Accessories', 200, 45.00),
(4, 'Monitor Stand', 'Furniture', 60, 55.00),
(5, 'Webcam HD', 'Electronics', 120, 69.99),
(6, 'Desk Lamp', 'Furniture', 90, 35.00),
(7, 'Mouse Pad XL', 'Accessories', 300, 19.99),
(8, 'Laptop Stand', 'Furniture', 75, 49.99);

In [ ]:
%%sql -r original_data
-- View original data (all 8 records)
SELECT * FROM PRODUCT_INVENTORY ORDER BY PRODUCT_ID;

In [ ]:
%%sql -r snapshot_time
-- Record the current query ID for Time Travel reference
SELECT CURRENT_TIMESTAMP() AS SNAPSHOT_TIME;

In [ ]:
%%sql -r set_ts
-- Save timestamp to a session variable
SET ts_before_delete = CURRENT_TIMESTAMP();

## Simulate Accidental Deletion

In [ ]:
%%sql -r accidental_delete
-- ACCIDENT: Someone accidentally deletes all 'Furniture' category products!
DELETE FROM PRODUCT_INVENTORY
WHERE CATEGORY = 'Furniture';

In [ ]:
%%sql -r after_delete
-- View current data - Furniture products are GONE
SELECT * FROM PRODUCT_INVENTORY ORDER BY PRODUCT_ID;

## Recovery Using Time Travel

In [ ]:
%%sql -r identify_deleted
-- Step 1: Identify what was deleted by comparing current vs historical data
SELECT * FROM PRODUCT_INVENTORY
    BEFORE (TIMESTAMP => $ts_before_delete)
WHERE PRODUCT_ID NOT IN (SELECT PRODUCT_ID FROM PRODUCT_INVENTORY)
ORDER BY PRODUCT_ID;

In [ ]:
%%sql -r recover_records
-- Step 2: Recover the deleted records by inserting them back from historical state
INSERT INTO PRODUCT_INVENTORY
    SELECT * FROM PRODUCT_INVENTORY
        BEFORE (TIMESTAMP => $ts_before_delete)
    WHERE PRODUCT_ID NOT IN (SELECT PRODUCT_ID FROM PRODUCT_INVENTORY);

In [ ]:
%%sql -r verify_recovery
-- Step 3: Verify recovery - all 8 records should be back
SELECT * FROM PRODUCT_INVENTORY ORDER BY PRODUCT_ID;

In [ ]:
%%sql -r count_check
-- Confirm record count matches original
SELECT 
    COUNT(*) AS CURRENT_COUNT,
    (SELECT COUNT(*) FROM PRODUCT_INVENTORY
        BEFORE (TIMESTAMP => $ts_before_delete)) AS ORIGINAL_COUNT
FROM PRODUCT_INVENTORY;